# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'
!rm -rf /kaggle/working/*

In [2]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 50.7 MB/s eta 0:00:00
dependencies ok


In [3]:
from pathlib import Path
import json, zipfile, hashlib, time
import numpy as np
import torch
import torch.nn as nn

try:
    import onnx
    import onnxruntime as ort
except Exception as e:
    raise ImportError('This notebook needs onnx and onnxruntime available in the Kaggle environment.') from e

# Prefer Kaggle working directory when available; otherwise use /mnt/data for local verification.
if Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working')
elif Path('/mnt/data').exists():
    WORK_DIR = Path('/mnt/data')
else:
    WORK_DIR = Path.cwd()

OUTPUT_DIR = WORK_DIR / 'task112_anchor_reflection_structural_out'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / 'task112.onnx'
REPORT_PATH = OUTPUT_DIR / 'task112_anchor_reflection_structural_verification_report.json'
SUBMISSION_ZIP = Path.cwd() / 'submission.zip'

def find_task_json():
    candidates = [
        Path.cwd() / 'task112.json',
        Path('/mnt/data/task112.json'),
        Path('/kaggle/working/task112.json'),
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('task112.json'))
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find task112.json. Put it next to this notebook or under /kaggle/input.')

TASK_JSON = find_task_json()
print('TASK_JSON =', TASK_JSON)
print('OUTPUT_DIR =', OUTPUT_DIR)

TASK_JSON = /kaggle/input/competitions/neurogolf-2026/task112.json
OUTPUT_DIR = /kaggle/working/task112_anchor_reflection_structural_out


In [4]:
with open(TASK_JSON) as f:
    task = json.load(f)

print({k: len(v) for k, v in task.items()})
for split in ['train', 'test', 'arc-gen']:
    ex = task[split][0]
    print(split, np.array(ex['input']).shape, np.array(ex['output']).shape)

{'train': 3, 'test': 1, 'arc-gen': 262}
train (20, 30) (20, 30)
test (18, 14) (18, 14)
arc-gen (12, 27) (12, 27)


In [5]:
def find_anchor(grid):
    g = np.array(grid, dtype=np.int64)
    for c in [int(v) for v in np.unique(g) if v != 0]:
        pts = np.argwhere(g == c)
        if len(pts) == 4:
            y0, x0 = pts.min(axis=0)
            y1, x1 = pts.max(axis=0)
            if y1 - y0 == 1 and x1 - x0 == 1 and np.all(g[y0:y0+2, x0:x0+2] == c):
                return int(c), int(y0), int(x0)
    return None, None, None

def solve_numpy(grid):
    g = np.array(grid, dtype=np.int64)
    out = g.copy()
    H, W = g.shape
    anchor_color, ay, ax = find_anchor(g)
    if anchor_color is None:
        return out
    for y, x in np.argwhere((g != 0) & (g != anchor_color)):
        c = int(g[y, x])
        for yy in (int(y), 2 * ay + 1 - int(y)):
            for xx in (int(x), 2 * ax + 1 - int(x)):
                if 0 <= yy < H and 0 <= xx < W and (out[yy, xx] == 0 or out[yy, xx] == c):
                    out[yy, xx] = c
    return out

def feature_key(ex):
    g = np.array(ex['input'], dtype=np.int64)
    H, W = g.shape
    anchor_color, ay, ax = find_anchor(g)
    payload = np.argwhere((g != 0) & (g != anchor_color)) if anchor_color is not None else np.empty((0, 2), int)
    if len(payload):
        py0, px0 = payload.min(axis=0)
        py1, px1 = payload.max(axis=0)
        cy = ay + 0.5
        cx = ax + 0.5
        qy = 'T' if payload[:, 0].mean() < cy else 'B'
        qx = 'L' if payload[:, 1].mean() < cx else 'R'
        payload_descriptor = (int(py1 - py0 + 1), int(px1 - px0 + 1), qy + qx)
        already_complete = int(np.array_equal(solve_numpy(g), g))
    else:
        payload_descriptor = (0, 0, 'none')
        already_complete = 0
    return (H // 5, W // 5, ay // 4 if ay is not None else -1, ax // 4 if ax is not None else -1, payload_descriptor, already_complete)

def split_arcgen_structurally(arc_examples):
    groups = {}
    for i, ex in enumerate(arc_examples):
        groups.setdefault(feature_key(ex), []).append(i)
    keys = sorted(groups, key=lambda k: repr(k))
    scored = sorted(keys, key=lambda k: hashlib.sha1(repr(k).encode()).hexdigest())
    target = max(1, round(0.30 * len(arc_examples)))
    train_idx, holdout_idx, holdout_keys = [], [], []
    n = 0
    for k in scored:
        if n < target:
            holdout_keys.append(k)
            holdout_idx.extend(groups[k])
            n += len(groups[k])
        else:
            train_idx.extend(groups[k])
    return train_idx, holdout_idx, holdout_keys, groups

def eval_rule(examples, indices=None):
    ok = total = 0
    bad = []
    seq = enumerate(examples) if indices is None else [(i, examples[i]) for i in indices]
    for i, ex in seq:
        pred = solve_numpy(ex['input'])
        tgt = np.array(ex['output'], dtype=np.int64)
        total += 1
        if np.array_equal(pred, tgt):
            ok += 1
        else:
            bad.append({'index': int(i), 'wrong_pixels': int((pred != tgt).sum())})
    return {'right': ok, 'total': total, 'bad': bad[:10]}

arc_train_idx, arc_holdout_idx, holdout_keys, groups = split_arcgen_structurally(task['arc-gen'])
print('arc-gen total:', len(task['arc-gen']))
print('structural train:', len(arc_train_idx))
print('structural holdout:', len(arc_holdout_idx))
print('structural groups:', len(groups), 'holdout groups:', len(holdout_keys))
print('visible rule:', eval_rule(task['train'] + task['test']))
print('arc-gen structural train rule:', eval_rule(task['arc-gen'], arc_train_idx))
print('arc-gen structural holdout rule:', eval_rule(task['arc-gen'], arc_holdout_idx))

arc-gen total: 262
structural train: 183
structural holdout: 79
structural groups: 199 holdout groups: 59
visible rule: {'right': 4, 'total': 4, 'bad': []}
arc-gen structural train rule: {'right': 183, 'total': 183, 'bad': []}
arc-gen structural holdout rule: {'right': 79, 'total': 79, 'bad': []}


In [6]:
class Task112AnchorReflectionModel(nn.Module):
    def __init__(self):
        super().__init__()
        banks = []
        for a in range(29):
            M = torch.zeros(30, 30, dtype=torch.float32)
            for i in range(30):
                j = 2 * a + 1 - i
                if 0 <= j < 30:
                    M[j, i] = 1.0
            banks.append(M)
        self.register_buffer('REFLECT_BANK', torch.stack(banks, dim=0))

    def forward(self, x):
        # x: [1,10,30,30], one-hot inside actual canvas and all-zero outside canvas.
        B = x.shape[0]
        nz = x[:, 1:, :, :]

        # Anchor color = non-background color whose total count is 4 and which forms a 2x2 solid block.
        counts = nz.sum(dim=(2, 3))
        count_is_four = ((counts > 3.5) & (counts < 4.5)).to(x.dtype)
        blocks = nz[:, :, :-1, :-1] * nz[:, :, 1:, :-1] * nz[:, :, :-1, 1:] * nz[:, :, 1:, 1:]
        has_block = (blocks.sum(dim=(2, 3)) > 0.5).to(x.dtype)
        anchor_color = count_is_four * has_block  # [B,9], one-hot over colors 1..9

        # Anchor top-left coordinate as one-hot over [29,29].
        anchor_tl = torch.clamp((blocks * anchor_color[:, :, None, None]).sum(dim=1, keepdim=True), 0, 1)
        row_w = torch.clamp(anchor_tl.sum(dim=3).squeeze(1), 0, 1)
        col_w = torch.clamp(anchor_tl.sum(dim=2).squeeze(1), 0, 1)

        # Reflection matrices chosen from precomputed banks.
        RY = (row_w[:, :, None, None] * self.REFLECT_BANK[None, :, :, :]).sum(dim=1)
        RX = (col_w[:, :, None, None] * self.REFLECT_BANK[None, :, :, :]).sum(dim=1)

        # Payload = every non-background channel except the anchor color.
        payload_ch = torch.cat([
            torch.zeros(B, 1, dtype=x.dtype, device=x.device),
            1.0 - torch.clamp(anchor_color, 0, 1)
        ], dim=1)
        P = x * payload_ch[:, :, None, None]

        # Reflect payload around anchor horizontal axis, vertical axis, and both axes.
        yref = torch.einsum('boi,bciw->bcow', RY, P)
        xref = torch.einsum('boj,bcyj->bcyo', RX, P)
        yxref = torch.einsum('boi,bcij->bcoj', RY, xref)
        added = torch.clamp(P + yref + xref + yxref, 0, 1)

        # Freeze existing nonzero input cells, including the anchor.
        orig_nonzero = torch.cat([torch.zeros_like(x[:, 0:1, :, :]), x[:, 1:, :, :]], dim=1)
        out_nonzero = torch.clamp(orig_nonzero + added, 0, 1)
        occupied = torch.clamp(out_nonzero[:, 1:, :, :].sum(dim=1, keepdim=True), 0, 1)
        out0 = x[:, 0:1, :, :] * (1.0 - occupied)
        return torch.cat([out0, out_nonzero[:, 1:, :, :]], dim=1)

model = Task112AnchorReflectionModel().eval()
print(model)

Task112AnchorReflectionModel()


In [7]:
def onehot30(grid):
    arr = np.array(grid, dtype=np.int64)
    x = np.zeros((1, 10, 30, 30), dtype=np.float32)
    H, W = arr.shape
    for r in range(H):
        for c in range(W):
            x[0, arr[r, c], r, c] = 1.0
    return x, H, W

def labels_from_output(y, H, W):
    return y[0, :, :H, :W].argmax(0).astype(np.int64)

def eval_torch(examples, indices=None):
    ok = total = 0
    bad = []
    seq = enumerate(examples) if indices is None else [(i, examples[i]) for i in indices]
    with torch.no_grad():
        for i, ex in seq:
            x, H, W = onehot30(ex['input'])
            y = model(torch.from_numpy(x)).numpy()
            pred = labels_from_output(y, H, W)
            tgt = np.array(ex['output'], dtype=np.int64)
            total += 1
            if np.array_equal(pred, tgt):
                ok += 1
            else:
                bad.append({'index': int(i), 'wrong_pixels': int((pred != tgt).sum())})
    return {'right': ok, 'total': total, 'bad': bad[:10]}

print('torch visible:', eval_torch(task['train'] + task['test']))
print('rule arc-gen structural holdout:', eval_rule(task['arc-gen'], arc_holdout_idx))

torch visible: {'right': 4, 'total': 4, 'bad': []}
rule arc-gen structural holdout: {'right': 79, 'total': 79, 'bad': []}


In [8]:
dummy = torch.zeros(1, 10, 30, 30, dtype=torch.float32)
dummy[:, 0, :, :] = 1.0

# `dynamo=False` uses the legacy exporter and avoids requiring onnxscript in some Kaggle environments.
torch.onnx.export(
    model, dummy, str(MODEL_PATH),
    opset_version=17,
    input_names=['input'], output_names=['output'],
    dynamic_axes=None, dynamo=False
)

onnx_model = onnx.load(str(MODEL_PATH))
onnx.checker.check_model(onnx_model)
ops = sorted(set(node.op_type for node in onnx_model.graph.node))
forbidden = ['Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function']
forbidden_present = [op for op in ops if op in forbidden]

def shape_of(value_info):
    return [d.dim_value for d in value_info.type.tensor_type.shape.dim]

input_shape = shape_of(onnx_model.graph.input[0])
output_shape = shape_of(onnx_model.graph.output[0])
size_bytes = MODEL_PATH.stat().st_size

print('MODEL_PATH:', MODEL_PATH)
print('size_bytes:', size_bytes)
print('input_shape:', input_shape)
print('output_shape:', output_shape)
print('ops:', ops)
print('forbidden_present:', forbidden_present)

assert input_shape == [1, 10, 30, 30]
assert output_shape == [1, 10, 30, 30]
assert size_bytes < 1_400_000
assert not forbidden_present

/tmp/ipykernel_16/1375266217.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


MODEL_PATH: /kaggle/working/task112_anchor_reflection_structural_out/task112.onnx
size_bytes: 119264
input_shape: [1, 10, 30, 30]
output_shape: [1, 10, 30, 30]
ops: ['Add', 'And', 'Cast', 'Clip', 'Concat', 'Constant', 'Einsum', 'Greater', 'Less', 'Mul', 'ReduceSum', 'Slice', 'Squeeze', 'Sub', 'Unsqueeze']
forbidden_present: []


In [9]:
sess = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])

def eval_onnx(examples, indices=None):
    ok = total = 0
    bad = []
    seq = enumerate(examples) if indices is None else [(i, examples[i]) for i in indices]
    for i, ex in seq:
        x, H, W = onehot30(ex['input'])
        y = sess.run(None, {'input': x})[0]
        pred = labels_from_output(y, H, W)
        tgt = np.array(ex['output'], dtype=np.int64)
        total += 1
        if np.array_equal(pred, tgt):
            ok += 1
        else:
            bad.append({'index': int(i), 'wrong_pixels': int((pred != tgt).sum())})
    return {'right': ok, 'total': total, 'bad': bad[:10]}

t0 = time.time()
onnx_train = eval_onnx(task['train'])
onnx_test = eval_onnx(task['test'])
onnx_arcgen = eval_onnx(task['arc-gen'])
onnx_holdout = eval_onnx(task['arc-gen'], arc_holdout_idx)
onnx_seconds = time.time() - t0

print('onnx train:', onnx_train)
print('onnx test:', onnx_test)
print('onnx arc-gen:', onnx_arcgen)
print('onnx structural holdout:', onnx_holdout)
print('onnx eval seconds:', onnx_seconds)

assert onnx_train['right'] == onnx_train['total']
assert onnx_test['right'] == onnx_test['total']
assert onnx_holdout['right'] == onnx_holdout['total']

onnx train: {'right': 3, 'total': 3, 'bad': []}
onnx test: {'right': 1, 'total': 1, 'bad': []}
onnx arc-gen: {'right': 262, 'total': 262, 'bad': []}
onnx structural holdout: {'right': 79, 'total': 79, 'bad': []}
onnx eval seconds: 0.1633772850036621


In [10]:
def permute_example(ex, mapping):
    inp = np.array(ex['input'])
    out = np.array(ex['output'])
    pin = np.vectorize(lambda v: mapping.get(int(v), int(v)))(inp)
    pout = np.vectorize(lambda v: mapping.get(int(v), int(v)))(out)
    return {'input': pin.tolist(), 'output': pout.tolist()}

# Extra color-invariance stress test: arc-gen uses fixed colors, so validate generated color permutations too.
perms = []
for payload in [1, 4, 5, 6, 7, 8, 9]:
    for anchor in [1, 4, 5, 6, 7, 8, 9]:
        if payload != anchor:
            perms.append({2: payload, 3: anchor})
perms = perms[:24]

color_ok = color_total = 0
for ex in task['train'] + task['test'] + task['arc-gen'][:20]:
    for mp in perms:
        pex = permute_example(ex, mp)
        pred = solve_numpy(pex['input'])
        tgt = np.array(pex['output'], dtype=np.int64)
        color_total += 1
        color_ok += int(np.array_equal(pred, tgt))

print('color permutation stress:', color_ok, '/', color_total)
assert color_ok == color_total

color permutation stress: 576 / 576


In [11]:
report = {
    'task': 'task112',
    'model': 'color_invariant_2x2_anchor_payload_reflection_orbit',
    'public_io': {'input': input_shape, 'output': output_shape},
    'validation_protocol': {
        'structural_key': 'height_bin,width_bin,anchor_row_bin,anchor_col_bin,payload_bbox,source_quadrant,already_complete',
        'arcgen_total': len(task['arc-gen']),
        'struct_train_count': len(arc_train_idx),
        'struct_holdout_count': len(arc_holdout_idx),
        'structural_group_count': len(groups),
        'holdout_group_count': len(holdout_keys),
        'color_permutation_stress_total': color_total,
    },
    'rule_validation': {
        'train': eval_rule(task['train']),
        'test': eval_rule(task['test']),
        'arcgen_struct_train': eval_rule(task['arc-gen'], arc_train_idx),
        'arcgen_struct_holdout': eval_rule(task['arc-gen'], arc_holdout_idx),
        'arcgen_all': eval_rule(task['arc-gen']),
        'color_permutation_stress': {'right': color_ok, 'total': color_total},
    },
    'onnx_validation': {
        'train': onnx_train,
        'test': onnx_test,
        'arcgen_struct_holdout': onnx_holdout,
        'arcgen_all': onnx_arcgen,
        'seconds': onnx_seconds,
    },
    'onnx': {
        'size_bytes': size_bytes,
        'ops': ops,
        'forbidden_ops': forbidden_present,
    }
}
REPORT_PATH.write_text(json.dumps(report, indent=2))

with zipfile.ZipFile(SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH, 'task112.onnx')

print('REPORT_PATH:', REPORT_PATH)
print('SUBMISSION_ZIP:', SUBMISSION_ZIP)
print(json.dumps(report, indent=2)[:2500])

REPORT_PATH: /kaggle/working/task112_anchor_reflection_structural_out/task112_anchor_reflection_structural_verification_report.json
SUBMISSION_ZIP: /kaggle/working/submission.zip
{
  "task": "task112",
  "model": "color_invariant_2x2_anchor_payload_reflection_orbit",
  "public_io": {
    "input": [
      1,
      10,
      30,
      30
    ],
    "output": [
      1,
      10,
      30,
      30
    ]
  },
  "validation_protocol": {
    "structural_key": "height_bin,width_bin,anchor_row_bin,anchor_col_bin,payload_bbox,source_quadrant,already_complete",
    "arcgen_total": 262,
    "struct_train_count": 183,
    "struct_holdout_count": 79,
    "structural_group_count": 199,
    "holdout_group_count": 59,
    "color_permutation_stress_total": 576
  },
  "rule_validation": {
    "train": {
      "right": 3,
      "total": 3,
      "bad": []
    },
    "test": {
      "right": 1,
      "total": 1,
      "bad": []
    },
    "arcgen_struct_train": {
      "right": 183,
      "total": 183,
 